# What this notebook is about

We will analyze several packet captures in detail:

- How a browser contacts a web server for downloading a file

- How a client obtains the destination Ethernet address of frames (ARP)

- How `ping` and `traceroute` work (ICMP)


# Preparation


## Install software

In [ ]:
!apt install tshark
!apt install bc

In [ ]:
!pip install scapy
from scapy.all import *

## Scapy: filtering packets from a capture

In [ ]:
def myscapy_with_port(pkts, portnum, proto=TCP):
  if proto not in [TCP, UDP]:
    print("Error: proto must be either TCP or UDP")
    return []
  result = []
  print(proto)
  for packet in pkts:
    if packet.haslayer(proto) and portnum in (packet[proto].sport, packet[proto].dport):
      result.append(packet)
  return result

def myscapy_with_protocol(pkts, protocols):
    result = []
    for packet in pkts:
      if any(packet.haslayer(protocol) for protocol in protocols):
        result.append(packet)
    return result

def myscapy_with_ip_address(pkts, target_ip):
    result = []
    for packet in pkts:
      if (packet.haslayer(IP) and target_ip in (packet[IP].src, packet[IP].dst)) or \
        (packet.haslayer(ARP) and target_ip in (packet[ARP].psrc, packet[ARP].pdst)):
        result.append(packet)
    return result

def myscapy_with_offsets(pkts, offsets):
    for packet in pkts:
      selected_packets = [pkts[pos-1] for pos in offsets if pos <= len(pkts)]
    return selected_packets

## Scapy: displaying a packet



In [ ]:
from scapy.all import *
import binascii

# ANSI escape codes for colors (https://gist.github.com/fnky/458719343aabd01cfb17a3a4f7296797)
COLOR_ETHER = '\033[91m'  # Red
COLOR_IP = '\033[94m'  # Blue
COLOR_TCP = '\033[33m'  # Brown
COLOR_UDP = '\033[92m'  # Green
#COLOR_ICMP = '\033[96m'  # Cyan
COLOR_ICMP = '\033[92m'  # Green
COLOR_ARP = '\033[95m'  # Yellow
ENDC = '\033[0m'  # End coloring

# Function to format and display packet sections in hex and return a dictionary with addresses of each layer
def display_packet_in_hex(packet):
    if not packet.haslayer(Ether):
      print("Error: Packet is not Ethernet.")
      return

    addresses = {}

    # Convert the whole packet to hex
    packet_hex = binascii.hexlify(bytes(packet)).decode()

    # Ethernet addresses
    eth_src = packet[Ether].src
    eth_dst = packet[Ether].dst
    addresses['Ethernet'] = {'Source': eth_src, 'Destination': eth_dst}
    # Ethernet Header
    eth_len = 14  # Ethernet header length is always 14 bytes
    eth_header = packet_hex[:eth_len * 2]  # Ethernet header in hex

    # Check if it's an IP packet
    if packet.haslayer(IP):
        ip_len = packet[IP].ihl * 4  # IP header length in bytes (ihl is in 4-byte words)

        # IP addresses
        ip_src = packet[IP].src
        ip_dst = packet[IP].dst
        addresses['IP'] = {'Source': ip_src, 'Destination': ip_dst}

        # IP Header
        ip_header = packet_hex[eth_len * 2:(eth_len + ip_len) * 2]  # IP header in hex

        # Check for TCP, UDP, or ICMP inside the IP payload
        if packet.haslayer(TCP):
            tcp_len = packet[TCP].dataofs * 4  # TCP header length in bytes
            tcp_header = packet_hex[(eth_len + ip_len) * 2:(eth_len + ip_len + tcp_len) * 2]
            remaining_payload = packet_hex[(eth_len + ip_len + tcp_len) * 2:]

            # TCP addresses (ports)
            tcp_src_port = packet[TCP].sport
            tcp_dst_port = packet[TCP].dport
            addresses['TCP'] = {'Source': str(tcp_src_port) + ' (' + tcp_header[0:4] + ')', 'Destination': str(tcp_dst_port) + ' (' +  tcp_header[4:8] +')'}
            addresses['Application'] = {'Payload': remaining_payload}

            tcp_payload = bytes(packet[TCP].payload)
            if b"HTTP" in tcp_payload or b"GET" in tcp_payload or b"POST" in tcp_payload:
              http_message = tcp_payload.decode('utf-8')
              addresses['Application'] = {'Payload': remaining_payload + '\n\n(Payload HTTP)\n' + http_message }
            else:
              addresses['Application'] = {'Payload': remaining_payload}

            print(f"{COLOR_ETHER}{eth_header}{ENDC}", end="")
            print(f"{COLOR_IP}{ip_header}{ENDC}", end="")
            print(f"{COLOR_TCP}{tcp_header}{ENDC}", end="")
            print(f"{remaining_payload}")

        elif packet.haslayer(UDP):
            udp_len = 8  # UDP header length is always 8 bytes
            udp_header = packet_hex[(eth_len + ip_len) * 2:(eth_len + ip_len + udp_len) * 2]
            remaining_payload = packet_hex[(eth_len + ip_len + udp_len) * 2:]

            # UDP addresses (ports)
            udp_src_port = packet[UDP].sport
            udp_dst_port = packet[UDP].dport
            addresses['UDP'] = {'Source': str(udp_src_port) + ' (' + udp_header[0:4] + ')', 'Destination': str(udp_dst_port) + ' (' +  udp_header[4:8] +')'}
            addresses['Application'] = {'Payload': remaining_payload}
            if packet.haslayer(DNS):
              dns_message = packet[DNS]
              addresses['Application'] = {'Payload': remaining_payload + '\n(' + dns_message.summary() +')'}
            else:
              addresses['Application'] = {'Payload': remaining_payload}

            print(f"{COLOR_ETHER}{eth_header}{ENDC}", end="")
            print(f"{COLOR_IP}{ip_header}{ENDC}", end="")
            print(f"{COLOR_UDP}{udp_header}{ENDC}", end="")
            print(f"{remaining_payload}")

        elif packet.haslayer(ICMP):
            icmp_len = 8  # ICMP header length is 8 bytes
            icmp_header = packet_hex[(eth_len + ip_len) * 2:(eth_len + ip_len + icmp_len) * 2]
            remaining_payload = packet_hex[(eth_len + ip_len + icmp_len) * 2:]

            # ICMP doesn't have ports but we can note it is ICMP
            addresses['ICMP'] = {'Type': packet[ICMP].type, 'Code': packet[ICMP].code}

            print(f"{COLOR_ETHER}{eth_header}{ENDC}", end="")
            print(f"{COLOR_IP}{ip_header}{ENDC}", end="")
            print(f"{COLOR_ICMP}{icmp_header}{ENDC}", end="")
            print(f"{remaining_payload}")

        else:
            print("Error: IP payload is neither TCP, UDP, nor ICMP.")

    # Check if it's an ARP packet
    elif packet.haslayer(ARP):
        arp_len = 28  # ARP packet length is 28 bytes

        # ARP addresses
        arp_src_eth = packet[ARP].hwsrc
        arp_dst_eth = packet[ARP].hwdst
        arp_src = packet[ARP].psrc
        arp_dst = packet[ARP].pdst
        addresses['ARP'] = {'Source': str(arp_src_eth) + ' / ' + str(arp_src), 'Destination': str(arp_dst_eth) + ' / ' + str(arp_dst)}

        arp_header = packet_hex[eth_len * 2:(eth_len + arp_len) * 2]  # ARP header in hex

        print(f"{COLOR_ETHER}{eth_header}{ENDC}", end="")
        print(f"{COLOR_ARP}{arp_header}{ENDC}", end="")
        print()

    else:
        print("Error: Ethernet payload is neither IP nor ARP.")

    # Return the dictionary with addresses
    return addresses

# Function to display simplified headers (as returned by display_packet_in_hex())
def explain_packet(packet_addresses):
  if "Ethernet" in packet_addresses:
    source = packet_addresses["Ethernet"]["Source"]
    destination = packet_addresses["Ethernet"]["Destination"]
    print("Ethernet: ", end="")
    print(f"{COLOR_ETHER}{source}--->{destination}{ENDC}")

  if "IP" in packet_addresses:
    source = packet_addresses["IP"]["Source"]
    destination = packet_addresses["IP"]["Destination"]
    print("IP: ", end="")
    print(f"{COLOR_IP}{source}--->{destination}{ENDC}")
  if "ARP" in packet_addresses:
    source = packet_addresses["ARP"]["Source"]
    destination = packet_addresses["ARP"]["Destination"]
    print("ARP: ", end="")
    print(f"{COLOR_ARP}{source}--->{destination}{ENDC}")

  if "TCP" in packet_addresses:
    source = packet_addresses["TCP"]["Source"]
    destination = packet_addresses["TCP"]["Destination"]
    print("TCP: ", end="")
    print(f"{COLOR_TCP}{source}--->{destination}{ENDC}")
  if "UDP" in packet_addresses:
    source = packet_addresses["UDP"]["Source"]
    destination = packet_addresses["UDP"]["Destination"]
    print("UDP: ", end="")
    print(f"{COLOR_UDP}{source}--->{destination}{ENDC}")
  if "ICMP" in packet_addresses:
    tt = packet_addresses["ICMP"]["Type"]
    print("ICMP: ", end="")
    print(f"{COLOR_ICMP}Type {tt}{ENDC}")
  if "Application" in packet_addresses:
    pp = packet_addresses["Application"]["Payload"]
    print("Application: ", end="")
    print(pp)

# Encapsulation (`curl`)

The capture `curl_http_units.pcapng` contains the execution of `curl -O http://www.units.it`. This command downloads the document with the specified URL and stores that document in a local file.

In [ ]:
!curl -O https://raw.githubusercontent.com/bartolialberto/ComputerNetworks/gh-pages/captures/curl_http_units.pcapng

Take a moment to think which frames will be contained in the capture.

Then, execute the following command that will print one line for each frame in the capture (see section "Tshark: Displaying all frames in compact form" below" for a description of the output format).


In [ ]:
!tshark -n -r curl_http_units.pcapng

You can see that frames correspond to the actions that we expected: the client sends a DNS request to the name server for resolving www.units.it. Then, the client sends an HTTP request to the corresponding IP address and receives the corresponding HTTP response.

Frames labelled as TCP that do not have any payload are beyond the scope of this course. In most cases they are used for opening and for closing TCP connections. In this capture, 5-7 are for opening the TCP connection while 10-14 are for closing.

Frames labelled as ARP are very important. They will be studied later.

Now execute the following code, that will print all the frames of our capture that carry DNS messages.

The first line (`display_packet_in_hex()`) prints the frame content in hex form, with different colors for each layer. The other lines, one for each layer, contain the address information of that layer: Ethernet addresses, IP addresses, port numbers.





In [ ]:
packets = rdpcap("curl_http_units.pcapng")
filtered_packets = myscapy_with_protocol(packets, [DNS])
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

* Which layers are present in each frame? Do they match your expectations?

* What is the IP address of the node that executes curl?

* What is the IP address of the name server?

* What is the Ethernet address of the node that executes curl?

* Can you tell the Ethernet address of the name server? Think carefully.

* Try to see where address information resides at each layer (you might want to use the "Convert" section in this notebook)

Now execute the following code, that will print the frames that carry the HTTP request and the HTTP response.

(The function used above to select only frames with a specific protocol cannot be used for this purpose, for reasons beyond the scope of this course; we have to specify the frames of our interest by their offset in the capture; we could have specified them by port number but in that case we would have obtained also the frames for opening and closing the TCP connection).


In [ ]:
filtered_packets = myscapy_with_offsets(packets, [8,9])
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)


* Which layers are present in each frame? Do they match your expectations?

* What is the IP address of the web server?

* Can you tell the Ethernet address of the web server? Think carefully.

* Try to see where address information resides at each layer (you might want to use the "Convert" section in this notebook)

# ARP

In [ ]:
!curl -O https://raw.githubusercontent.com/bartolialberto/ComputerNetworks/gh-pages/captures/curl_http_units.pcapng

The capture `curl_http_units.pcapng` contains the execution of `curl -O http://www.units.it`. This command downloads the document with the specified URL and stores that document in a local file.

Execute the following command.



In [ ]:
!tshark -n -r curl_http_units.pcapng

Focus on the IP addresses, in particular by looking at frames 3 and 4 (DNS request and DNS response) and at frames 8 and 9 (HTTP request and HTTP response). Let C denote the node that executes `curl`.

* What is the IP address of C?

* C is in the same network as the name server?

* C is in the same network as the web server?

Now execute the next cell to see the content of DNS and HTTP messages in detail.

In [ ]:
filtered_packets = myscapy_with_offsets(packets, [3,4,8,9])
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

* What is the Ethernet address of C?

* Why does C send the DNS request and the HTTP request to the *same* destination Ethernet address?

* How did C obtained that Ethernet address?

* Why does C receivesì the DNS response and the HTTP response from the *same* source Ethernet address?

* Try to see where address information resides at each layer (you might want to use the "Convert" section in this notebook)

Look again at the full listing of all the frames, a few cells above. You can see that the capture begins with an ARP request and the matching ARP response. The next code cell displays the content of those frames in detail.




In [ ]:
filtered_packets = myscapy_with_offsets(packets, [1,2])
# it might be also like this:
# filtered_packets = myscapy_with_protocol(packets, [ARP])
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

* Who is sending the ARP request? C or some other node?

* What is the IP address for which the matching Ethernet address is requested?

* Why C is requesting that IP address? How did C obtained that IP address?

* Does the ARP response contain the Ethernet address involved in the DNS/HTTP exchanges of the previous code cell?

Then, try to identify in the hex display of the frames:

*   The four fields of each ARP message (you may want to use the "Convert" section).

*   The type field of each Ethernet frame.

The next cell displays a capture containing only a gratuitous ARP (capture made publicly available by [Chris Sanders](https://github.com/chrissanders/packets)).

*   Why is this ARP request a "gratuitous" ARP?

In [ ]:
!curl -O https://raw.githubusercontent.com/chrissanders/packets/master/arp_gratuitous.pcapng
print('-' * 20)
cs_packets = rdpcap("arp_gratuitous.pcapng")
for packet in cs_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

# ICMP

### ping IP address

In [ ]:
!curl -O https://raw.githubusercontent.com/bartolialberto/ComputerNetworks/gh-pages/captures/ping_ip.pcapng

The capture `ping_ip.pcapng` contains the execution of `ping 10.0.2.4`.

Take a moment to think which frames will be contained in the capture.

Then, execute the following command that will print one line for each frame in the capture (see section "Tshark: Displaying all frames in compact form" below" for a description of the output format).

* Does the frame list match your expectations? If not, why?

In [ ]:
!tshark -n -r ping_ip.pcapng

Now execute the following code, that will print all the frames of our capture.

The textual meaning of ICMP types can be found at [this link](https://www.iana.org/assignments/icmp-parameters/icmp-parameters.xhtml).



In [ ]:
packets = rdpcap("ping_ip.pcapng")
for packet in packets:
    addresses = display_packet_in_hex(packet)
    explain_packet(addresses)
    print('-' * 20)

* Which layers are present in each frame? Do they match your expectations?

Let S be the node that executes `ping` and let R be the "ping'ed" node.

* What are the respective IP addresses?

* What are the respective Ethernet addresses?

* How did S obtain the Ethernet address of R?

* Are S and R in the same network?

Try to see where address information resides at each layer (you might want to use the "Convert" section in this notebook)

### ping DNS name

In [ ]:
!curl -O https://raw.githubusercontent.com/bartolialberto/ComputerNetworks/gh-pages/captures/ping_realmadrid.pcapng

The capture `ping_realmadrid.pcapng` contains the execution of `ping www.realmadrid.es`.

Take a moment to think which frames will be contained in the capture.

Then, execute the following command that will print one line for each frame in the capture (see section "Tshark: Displaying all frames in compact form" below" for a description of the output format).

* Does the frame list match your expectations? Probably you are surprised by frames 1 and 2. Can you explain their presence?

In [ ]:
!tshark -n -r ping_realmadrid.pcapng

Now execute the following code, that will print all the frames of our capture (skipping frames 1, 2, 9, 10).




In [ ]:
packets = rdpcap("ping_realmadrid.pcapng")
filtered_packets = myscapy_with_offsets(packets, list(range(3,9)))
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

Let S be the node that executes `ping` and let R be the "ping'ed" node (`www.realmadrid.es`).

* How many different IP addresses appear in this capture?

* How many different Ethernet addresses appear in this capture?

* Why packets addressed at *different* IP addresses are transported by frames all addressed at the *same* Ethernet address? In other words, why all the packets are addressed to only one Ethernet address?

* How did S obtained that Ethernet address?

## traceroute IP address

The next cell displays a capture containing an execution of traceroute (capture made publicly available by Chris Sanders).

IMPORTANT: the capture does not contain any ARP frame (either the capture was collected at an instant when the ARP cache already contained the necessary information, or the ARP traffic was discarded before saving the capture).

In [ ]:
!curl -O https://raw.githubusercontent.com/chrissanders/packets/master/icmp_traceroute.pcapng

Take a moment to think which frames will be contained in the capture.

Then, execute the following command that will print one line for each frame in the capture (see section "Tshark: Displaying all frames in compact form" below" for a description of the output format).

* Does the frame list match your expectations?

* How many "iterations" are executed for each step?

In [ ]:
!tshark -n -r icmp_traceroute.pcapng


Execute the next cell, that prints the content of one ICMP response for some of the routers along the path from the destination.

The textual meaning of ICMP types can be found at [this link](https://www.iana.org/assignments/icmp-parameters/icmp-parameters.xhtml).


In [ ]:
packets = rdpcap("icmp_traceroute.pcapng")
filtered_packets = myscapy_with_offsets(packets, [2,8,14,20,26,32,38])
for packet in filtered_packets:
   addresses = display_packet_in_hex(packet)
   explain_packet(addresses)
   print('-' * 20)

* What are the IP addresses of those routers?

* Why all the frames are sent from the *same* Ethernet address?

# Utilities

## Scapy Examples

In [ ]:
filtered_pkts = myscapy_with_port(packets, 53, proto=UDP)
for p in filtered_pkts:
  addresses = display_packet_in_hex(p)
  explain_packet(addresses)
  print('-' * 20)

In [ ]:
filtered_pkts = myscapy_with_protocol(packets, [ARP, TCP])
for p in filtered_pkts:
  addresses = display_packet_in_hex(p)
  explain_packet(addresses)
  print('-' * 20)

In [ ]:
filtered_pkts = myscapy_with_ip_address(packets, '172.16.0.122')
for p in filtered_pkts:
  addresses = display_packet_in_hex(p)
  explain_packet(addresses)
  print('-' * 20)

## Convert to/from HEX and BIN

hex ---> dec (replace FF with the hex number you want to convert)

In [ ]:
!echo "ibase=16; FF" | bc

dec ---> hex (replace 192 with the hex number you want to convert)

In [ ]:
!echo "obase=16; 192" | bc

bin ---> dec (replace 1101 with the hex number you want to convert)

In [ ]:
!echo "ibase=2; 1101" | bc

dec ---> bin (replace 192 with the hex number you want to convert)

In [ ]:
!echo "obase=2; 192" | bc

## Tshark

**tshark** is a very powerful (and complex) tool for displaying in a human-readable way **every field of every protocol**.

tshark is a command-line version of wireshark; it can be used for capturing frames, filtering frames, displaying frames in a human-readable way; in colab notebooks it cannot be used for capturing frames.

If you want to have a detailed explanation of the options used in each of the commands below, just ask Gemini at the right of this window (the Google AI assistant).

The "Convert" section may be useful for understanding the content of the frames.

### Displaying all frames in compact form

The basic form of the command displays one line for each frame in the capture.

The first field is the number of the frame in the capture, the second field is the time at which the frame was sent or received (since the beginning of the capture).

The third field indicates "the most important" protocol in the frame. The fourth field is irrelevant to us. The remaining part of the line is a textual summary.

In [ ]:
!tshark -n -r curl_http_units.pcapng

### Displaying frames in detail

Display each frame in hex and in ASCII (obviously, the ASCII representation is useful only for those parts of a frame that do contain ASCII data).

In [ ]:
!tshark -n -r curl_http_units.pcapng -x

With a verbose explanation of each single field.

In [ ]:
!tshark -n -r arp_resolution.pcapng -x -V

### Choosing which frames to display

#### Only frames that contain a specified protocol

In [ ]:
!tshark -n -r curl_http_units.pcapng -Y "(dns) or (arp)" -x -V

#### Only frames with a specified port number

In [ ]:
!tshark -n -r curl_http_units.pcapng -Y "tcp.port == 80" -x

In [ ]:
!tshark -n -r curl_http_units.pcapng -Y "udp.port == 53" -x

#### Only the *n*-th frame

In [ ]:
!tshark -r curl_http_units.pcapng -Y 'frame.number == 9' -x -V

### Listing all connections

In [ ]:
!curl -O https://raw.githubusercontent.com/chrissanders/packets/master/http_espn.pcapng

#### All TCP connections, no duplicates

This command lists all the TCP connections in the capture. Each connection appears only once. The order in the listing does not correspond to the order in which connections appear in the capture.

In [ ]:
!tshark -r http_espn.pcapng -T fields -e ip.src -e ip.dst -e tcp.srcport -e tcp.dstport -Y 'tcp' -n | \
awk '{ if ($1 < $2 || ($1 == $2 && $3 < $4)) print $1":"$3 " -> " $2":"$4 }' | \
sort | uniq


#### All UDP conversations (analogous to TCP connections)

The same as above

In [ ]:
!tshark -r http_espn.pcapng -T fields -e ip.src -e ip.dst -e udp.srcport -e udp.dstport -Y 'udp' -n | \
awk '{ if ($1 < $2 || ($1 == $2 && $3 < $4)) print $1":"$3 " -> " $2":"$4 }' | \
sort | uniq

